# Chapter 6 Metrics

Computes the metrics of chapter 6 (RQ3) that are not printed in `attack_model_training*.ipynb` and `attack_day_model_backtest.ipynb` yet.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/content/drive/MyDrive/master-thesis").resolve()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
RESULTS_DIR = NOTEBOOK_DIR / 'results_targeted/attack_day_model'
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from scr.attack_day_model.preprocessing import (chronological_train_test_split,
                                                TECHNICAL_FEATURE_COLUMNS, ENGINEERED_FEATURE_NAMES)
from scr.attack_day_model.model import load_X, load_y, load_model

logging.basicConfig(level=logging.ERROR, force=True)
logger = logging.getLogger("chapter6_metrics")
pd.set_option("display.precision", 3)

REF_NAME, AUG_NAME = "attack_dataset_(GOOGL)", "attack_dataset_1"
REF_DIR, AUG_DIR = PROJECT_ROOT / f"data/{REF_NAME}", PROJECT_ROOT / f"data/{AUG_NAME}"
MODEL_DIR = NOTEBOOK_DIR / "models/attack_day"
REF_MODEL = (MODEL_DIR / "attack_day_googl.pkl", MODEL_DIR / "metadata_googl.json")
AUG_MODEL = (MODEL_DIR / "attack_day.pkl", MODEL_DIR / "metadata.json")
BACKTEST_DIR = NOTEBOOK_DIR / "results_targeted/attack_day_model"

def show(label, value):
    v = f"{value:,}" if isinstance(value, (int, np.integer)) else f"{value:.3f}"
    print(f"{label:<58s} {v:>8s}")

def load_dataset(ds_dir, name, ticker=None):
    """Same filtering as the training notebooks; keeps ticker, strategy and calendar aside."""
    out = {}
    for split in ("train", "test"):
        X = load_X(ds_dir / f"{name}_X_{split}.csv", logger)
        y = load_y(ds_dir / f"{name}_y_{split}.csv", logger)
        tick = X[[c for c in X if c.startswith("attacked_ticker_")]].idxmax(axis=1).str.replace("attacked_ticker_", "")
        keep = (tick == ticker) if ticker else (tick != "AMZN")
        info = pd.DataFrame({"ticker": tick, "strategy": np.where(X["strategy_ROC"] == 1, "ROC", "MA"),
                             "force_buy": X["objective_force_buy"], "day_of_week": X["day_of_week"],
                             "quarter": X["quarter"]})[keep].reset_index(drop=True)
        X = X[keep].reset_index(drop=True)
        drop = ["attack_day", "attack_date"] + [c for c in X if c.startswith(
            ("setup_", "model_", "strategy_", "attacked_ticker_", "timing_policy_"))]
        out[f"X_{split}"], out[f"y_{split}"], out[f"info_{split}"] = X.drop(columns=drop, errors="ignore"), y[keep].reset_index(drop=True), info
    out["n_fit"] = int(len(out["X_train"]) * 0.8)
    return out

def load_bundle(paths):
    return load_model(paths[0], logger), json.load(open(paths[1], encoding="utf-8"))

ref, aug = load_dataset(REF_DIR, REF_NAME, "GOOGL"), load_dataset(AUG_DIR, AUG_NAME)
(ref_model, ref_meta), (aug_model, aug_meta) = load_bundle(REF_MODEL), load_bundle(AUG_MODEL)

Mounted at /content/drive


## 1 - Dataset (6.1 and 6.2)

In [2]:
for name, d in (("Reference", ref), ("Augmented", aug)):
    y_all = pd.concat([d["y_train"], d["y_test"]])
    show(f"{name} dataset: rows", int(len(y_all)))
    show(f"{name} dataset: positive rows", int(y_all.sum()))
    show(f"{name} dataset: share positive", y_all.mean())
    show(f"{name}: training share positive", d["y_train"].mean())
    show(f"{name}: test share positive", d["y_test"].mean())
    show(f"{name}: validation share positive", d["y_train"].iloc[d["n_fit"]:].mean())
for split in ("train", "test"):
    m = aug[f"info_{split}"]["ticker"] == "AAPL"
    show(f"AAPL rows ({split})", int(m.sum()))
    show(f"AAPL share positive ({split})", aug[f"y_{split}"][m].mean())

feat = pd.read_csv(AUG_DIR / f"{AUG_NAME}_features.csv",
                   usecols=lambda c: c in {"attack_day", "attack_date", "attacked_ticker_GOOGL"})
feat["attack_date"] = pd.to_datetime(feat["attack_date"])
for name, part in zip(("Training", "Test"), chronological_train_test_split(feat)):
    g = part[part["attacked_ticker_GOOGL"] == 1]["attack_date"]
    print(f"{name} period: {g.min().date()} -> {g.max().date()}")

print("\nShare of vulnerable attacks by stock and strategy (training set):")
display(aug["y_train"].groupby([aug["info_train"]["ticker"], aug["info_train"]["strategy"]]).mean().unstack())

Reference dataset: rows                                       7,811
Reference dataset: positive rows                                834
Reference dataset: share positive                             0.107
Reference: training share positive                            0.109
Reference: test share positive                                0.098
Reference: validation share positive                          0.142
Augmented dataset: rows                                      15,489
Augmented dataset: positive rows                              1,609
Augmented dataset: share positive                             0.104
Augmented: training share positive                            0.108
Augmented: test share positive                                0.087
Augmented: validation share positive                          0.123
AAPL rows (train)                                             6,104
AAPL share positive (train)                                   0.108
AAPL rows (test)                                

strategy,MA,ROC
ticker,,
AAPL,0.020,0.183
GOOGL,0.049,0.160


## 2 - Reference classifier (6.1)

In [3]:
comp = pd.read_csv(REF_DIR / f"{REF_NAME}_model_comparison.csv")
ens = comp[comp["name"] != "logreg_class_weight"]
show("Ensembles: max difference of CV PR-AUC", ens["cv_pr_auc_mean"].max() - ens["cv_pr_auc_mean"].min())
show("Ensembles: CV PR-AUC std, min", ens["cv_pr_auc_std"].min())
show("Ensembles: CV PR-AUC std, max", ens["cv_pr_auc_std"].max())
best = comp.iloc[0]
prev_val = ref["y_train"].iloc[ref["n_fit"]:].mean()
show("Validation PR-AUC / prevalence (selected)", best["val_pr_auc"] / prev_val)

y = ref["y_test"]
p = ref_model.predict_proba(ref["X_test"][ref_meta["feature_columns"]])[:, 1]
flag = p >= float(ref_meta["threshold"])
show("Test PR-AUC / prevalence", average_precision_score(y, p) / y.mean())
show("Test: share of attacks flagged", flag.mean())

rate = ref["y_train"].groupby(ref["info_train"]["force_buy"]).mean()
score = ref["info_test"]["force_buy"].map(rate)
show("Objective rule: PR-AUC / prevalence", average_precision_score(y, score) / y.mean())

Ensembles: max difference of CV PR-AUC                        0.013
Ensembles: CV PR-AUC std, min                                 0.085
Ensembles: CV PR-AUC std, max                                 0.096
Validation PR-AUC / prevalence (selected)                     1.484
Test PR-AUC / prevalence                                      1.694
Test: share of attacks flagged                                0.364
Objective rule: PR-AUC / prevalence                           1.293


In [4]:
CALENDAR = ["day_of_week", "month", "quarter", "is_monday", "is_friday", "is_quarter_end_month"]
def group(f):
    if f.startswith("objective_"): return "Objective"
    if f in CALENDAR: return "Calendar"
    if f.startswith("ohlcv_lag_"): return "Price history"
    if f in TECHNICAL_FEATURE_COLUMNS: return "Technical indicators"
    return "Derived" if f in ENGINEERED_FEATURE_NAMES else "Other"

def importance(model, meta):
    imp = pd.DataFrame({"feature": meta["feature_columns"], "importance": model.named_steps["clf"].feature_importances_})
    imp["group"] = imp["feature"].map(group)
    return imp.sort_values("importance", ascending=False).reset_index(drop=True)

imp = importance(ref_model, ref_meta)
display(imp.groupby("group")["importance"].sum())
show("Sum of the 10 most important features", imp["importance"].head(10).sum())
show("Sum of the 20 most important features", imp["importance"].head(20).sum())
print("Calendar features among the 20 most important:", imp.head(20).query("group == 'Calendar'")["feature"].tolist())
show("Rank of vulnerability_score", int(imp.index[imp["feature"] == "vulnerability_score"][0]) + 1)

,importance
group,
Calendar,0.020
Derived,0.166
Objective,0.113
Price history,0.544
Technical indicators,0.157


Sum of the 10 most important features                         0.304
Sum of the 20 most important features                         0.413
Calendar features among the 20 most important: ['month']
Rank of vulnerability_score                                      53


## 3 - Augmented Dataset (6.2)

In [5]:
yb = aug["y_test"]
rows = []
for mname, (model, meta) in {"GOOGL": (ref_model, ref_meta), "GOOGL + AAPL": (aug_model, aug_meta)}.items():
    for subset in ("GOOGL", "AAPL", "All"):
        m = np.ones(len(yb), bool) if subset == "All" else (aug["info_test"]["ticker"] == subset).to_numpy()
        p = model.predict_proba(aug["X_test"].loc[m, meta["feature_columns"]])[:, 1]
        rows.append({"training data": mname, "test attacks": subset,
                     "PR-AUC / prevalence": average_precision_score(yb[m], p) / yb[m].mean(),
                     "share flagged": (p >= float(meta["threshold"])).mean()})
display(pd.DataFrame(rows))

rate = aug["y_train"].groupby(aug["info_train"]["force_buy"]).mean()
score, rule = aug["info_test"]["force_buy"].map(rate), aug["info_test"]["force_buy"]
show("Objective rule on all test attacks: F1", f1_score(yb, rule))
show("Objective rule on all test attacks: ROC-AUC", roc_auc_score(yb, score))
show("Objective rule on all test attacks: PR-AUC", average_precision_score(yb, score))

,training data,test attacks,PR-AUC / prevalence,share flagged
0,GOOGL,GOOGL,1.694,0.364
1,GOOGL,AAPL,1.780,0.314
2,GOOGL,All,1.721,0.339
3,GOOGL + AAPL,GOOGL,1.589,0.615
4,GOOGL + AAPL,AAPL,1.613,0.645
5,GOOGL + AAPL,All,1.590,0.630


Objective rule on all test attacks: F1                        0.229
Objective rule on all test attacks: ROC-AUC                   0.616
Objective rule on all test attacks: PR-AUC                    0.118


In [6]:
g = importance(aug_model, aug_meta).groupby("group")["importance"].sum()
show("Augmented importance: objective", g["Objective"])
show("Augmented importance: calendar", g["Calendar"])
show("Augmented importance: other market features", g.drop(["Objective", "Calendar"]).sum())

for col in ("day_of_week", "quarter"):
    print(f"\nShare of vulnerable attacks by {col} (augmented dataset):")
    display(pd.DataFrame({"training": aug["y_train"].groupby(aug["info_train"][col]).mean(),
                          "test": yb.groupby(aug["info_test"][col]).mean()}))

Augmented importance: objective                               0.306
Augmented importance: calendar                                0.423
Augmented importance: other market features                   0.270

Share of vulnerable attacks by day_of_week (augmented dataset):


,training,test
day_of_week,,
0,0.106,0.086
1,0.107,0.093
2,0.118,0.091
3,0.109,0.087
4,0.101,0.076



Share of vulnerable attacks by quarter (augmented dataset):


,training,test
quarter,,
1,0.117,0.122
2,0.101,0.053
3,0.126,0.084
4,0.088,0.088


## 4 - The classifier as timing policy (6.3)

In [7]:
bt = pd.read_csv(BACKTEST_DIR / "attack_day_model_backtest.csv")
b = lambda s: s.astype(str).str.lower().isin({"true", "1", "yes"})
bt = bt[b(bt["valid"])].copy()
bt["trivial"] = b(bt["already_target"])
bt["nt"] = ~bt["trivial"]
bt["succ"] = b(bt["success"]) & bt["nt"]
bt["harm"] = bt["succ"] & (pd.to_numeric(bt["delta_cumulative_return_in_period"], errors="coerce") < 0)
bt["regime"] = np.where(bt["setup"].str.contains("rolling|cumulative"), "periodic", "fixed")

pol = bt.groupby("timing_policy").agg(trials=("nt", "size"), non_trivial=("nt", "sum"),
                                      successes=("succ", "sum"), harmful=("harm", "sum"))
pol["P_harm"] = pol["harmful"] / pol["non_trivial"]
pol["harmful / launched"] = pol["harmful"] / pol["trials"]
display(pol)

h = bt[bt["timing_policy"] != "attack_day_model"]
show("Six heuristic policies: non-trivial trials", int(h["nt"].sum()))
show("Six heuristic policies: successes", int(h["succ"].sum()))
show("Six heuristic policies: harmful", int(h["harm"].sum()))

clf = bt[bt["timing_policy"] == "attack_day_model"]
print("\nClassifier: trivial trials per objective")
display(clf.groupby("objective")["trivial"].agg(trivial="sum", trials="size"))
print("Non-trivial trials and successes per regime")
display(bt[bt["timing_policy"].isin(["attack_day_model", "uniform", "near_strategy_boundary"])]
        .groupby(["timing_policy", "regime"]).agg(non_trivial=("nt", "sum"), successes=("succ", "sum")))

,trials,non_trivial,successes,harmful,P_harm,harmful / launched
timing_policy,,,,,,
attack_day_model,96,24,5,0,0.000,0.000
high_pred_volatility,96,49,4,2,0.041,0.021
high_volatility,96,48,4,2,0.042,0.021
near_strategy_boundary,96,75,46,12,0.160,0.125
strong_trend,96,48,2,0,0.000,0.000
uniform,96,63,18,6,0.095,0.062
weak_trend,96,66,17,4,0.061,0.042


Six heuristic policies: non-trivial trials                      349
Six heuristic policies: successes                                91
Six heuristic policies: harmful                                  26

Classifier: trivial trials per objective


,trivial,trials
objective,,
force_buy,45,48
force_sell,27,48


Non-trivial trials and successes per regime


non_trivial  successes
timing_policy          regime                          
attack_day_model       fixed              10          5
                       periodic           14          0
near_strategy_boundary fixed              36         31
                       periodic           39         15
uniform                fixed              28         17
                       periodic           35          1

In [8]:
sel = pd.read_csv(BACKTEST_DIR / "attack_day_model_selected_days.csv")
print("Probability of the selected days:")
display(sel.groupby("objective")["attack_probability"].agg(["min", "max"]))
show("Selected trials above the threshold", int((sel["attack_probability"] >= float(ref_meta["threshold"])).sum()))

MAX_GAP = 3   # days closer than this (trading days) belong to the same market episode
for obj, d in sel.drop_duplicates(["objective", "attack_day"]).groupby("objective"):
    days = np.sort(d["attack_day"].to_numpy())
    show(f"{obj}: distinct market episodes", 1 + int((np.diff(days) > MAX_GAP).sum()))

Probability of the selected days:


,min,max
objective,,
force_buy,0.755,0.785
force_sell,0.537,0.606


Selected trials above the threshold                              96
force_buy: distinct market episodes                               6
force_sell: distinct market episodes                              6


## 5 - Classifier policy outcomes

In [ ]:
try:
    trials = results.copy()
except NameError:
    trials = pd.read_csv(RESULTS_DIR / "attack_day_model_backtest.csv")

def as_bool(s):
    return s.astype(str).str.lower().isin({"true", "1", "yes"})

mpl.rcParams.update({
    "font.size": 10, "axes.labelsize": 10, "legend.fontsize": 9,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "pdf.fonttype": 42,
})

as_bool = lambda s: s.astype(str).str.lower().isin({"true", "1", "yes"})
trials = trials[as_bool(trials["valid"])].copy()
trials["success"] = as_bool(trials["success"])
trials["already_target"] = as_bool(trials["already_target"])
trials["nontrivial"] = ~trials["already_target"]
trials["nt_success"] = trials["success"] & trials["nontrivial"]
trials["harmful"] = trials["nt_success"] & (trials["delta_cumulative_return_in_period"] < 0)

# ordered by the knowledge they require, as in Chapter 5
POLICY_ORDER = ["uniform", "high_volatility", "strong_trend", "weak_trend",
                "attack_day_model", "high_pred_volatility", "near_strategy_boundary"]
POLICY_NAMES = {"attack_day_model": "classifier"}

# Sanity check against Table tab:classifier-policy
check = trials.groupby("timing_policy").agg(
    trials=("nontrivial", "size"), trivial=("already_target", "sum"),
    S=("nt_success", "sum"), E=("harmful", "sum")).reindex(POLICY_ORDER)
print(check)   # expected for the classifier: 96 / 72 / 5 / 0

def outcome_shares(df):
    n = len(df)
    trivial = df["already_target"].sum()
    harmful = df["harmful"].sum()
    forced_no_loss = df["nt_success"].sum() - harmful
    failed = n - trivial - df["nt_success"].sum()
    return n, n - trivial, np.array([trivial, failed, forced_no_loss, harmful]) / n

parts = ["Trivial (target already emitted)", "Attack failed",
         "Signal forced, no loss in 20 days", "Signal forced and loss in 20 days"]
part_colors = ["#f2f2f2", "#d9d9d9", "#9ecae1", "#c0392b"]
part_hatch = ["////", None, None, None]
part_text = ["black", "black", "black", "white"]

fig, ax = plt.subplots(figsize=(6.3, 3.6))
for i, p in enumerate(POLICY_ORDER):
    n, n_nt, shares = outcome_shares(trials[trials["timing_policy"] == p])
    left = 0.0
    for share, color, hatch, txt in zip(shares, part_colors, part_hatch, part_text):
        ax.barh(i, share * 100, left=left, color=color, height=0.62,
                edgecolor="#bdbdbd" if hatch else "white", hatch=hatch, linewidth=0.5 if hatch else 1)
        count = int(round(share * n))
        if share * 100 >= 6:   # number of trials inside the segment, only where it fits
            ax.text(left + share * 50, i, f"{count}", va="center", ha="center", fontsize=8, color=txt,
                    bbox=dict(boxstyle="square,pad=0.1", facecolor=color, edgecolor="none") if hatch else None)
        left += share * 100
    # share of harmful attacks over all the attacks launched
    ax.text(101, i, f"{shares[3] * 100:.1f}%", va="center", ha="left", fontsize=8, color="#c0392b")

ax.set_yticks(range(len(POLICY_ORDER)))
ax.set_yticklabels([POLICY_NAMES.get(p, p) for p in POLICY_ORDER])
for tick, p in zip(ax.get_yticklabels(), POLICY_ORDER):
    if p == "attack_day_model":
        tick.set_fontweight("bold")
ax.invert_yaxis()
ax.set_xlim(0, 112)
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"])
ax.set_xlabel("Share of the trials launched")
ax.spines[["top", "right"]].set_visible(False)

handles = [plt.Rectangle((0, 0), 1, 1, facecolor=c, hatch=h, edgecolor="#bdbdbd" if h else c)
           for c, h in zip(part_colors, part_hatch)]
fig.legend(handles, parts, loc="lower center", ncol=2, frameon=False,
           bbox_to_anchor=(0.5, -0.04), fontsize=8)
fig.tight_layout(rect=(0, 0.11, 1, 1))
plt.show()